# Vajra — protect an image against AI editing (the real attack)

This notebook runs the attack from *Raising the Cost of Malicious AI-Powered Image Editing* (arXiv:2302.06588):
it optimises a perturbation that is invisible to you but that drags the image's **Stable Diffusion VAE latent** towards a flat grey target, so a diffusion model conditioned on your photo sees almost nothing to preserve.

**Why not on a phone?** The attack needs real gradients through the VAE encoder. A browser or an app cannot do that, so the Vajra app uses a cheap linear stand-in that — measured — moves the real latent by about 0.4% of the distance it should close. This notebook is the version that works: ~91% in eight steps.

**It is a deterrent, not a guarantee.** It raises the cost of an edit. Someone determined can crop, denoise or re-photograph the image; each of those costs them quality.

Runtime → Change runtime type → **T4 GPU** (free tier is fine). Then Runtime → Run all.

In [ ]:
# 1. Install the pieces (about a minute on Colab)
!pip install -q diffusers transformers safetensors pillow
!git clone -q https://github.com/merry8989-uk/vajra-image-guard.git
import torch, sys
sys.path.insert(0, 'vajra-image-guard/research')
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('running on', 'GPU' if torch.cuda.is_available() else 'CPU — switch the runtime to T4 for speed')

In [ ]:
# 2. Upload the photo you want to protect
from google.colab import files
uploaded = files.upload()
INPUT = next(iter(uploaded))
print('using', INPUT)

In [ ]:
# 3. Settings — eps is the budget: 0.12 is about 15/255 per channel, still invisible.
STEPS = 200      # 200 is the paper's setting; 60 already looks convincing
EPS = 0.12
SIZE = 512       # working resolution; larger is slower
GREY = 0.5       # the neutral target

In [ ]:
# 4. Run the attack
import importlib, sd_encoder_attack as sa
importlib.reload(sa)
from PIL import Image, ImageOps

img = ImageOps.exif_transpose(Image.open(INPUT)).convert('RGB')
vae = sa.load_vae('stabilityai/sd-vae-ft-mse', 'cuda' if __import__('torch').cuda.is_available() else 'cpu')
protected, stats = sa.run(img, vae=vae, steps=STEPS, eps=EPS, size=SIZE, grey=GREY, log=print)

OUT = INPUT.rsplit('.', 1)[0] + '-vajra.png'
protected.save(OUT)
print()
print(f'latent distance to grey: {stats["latent_before"]:.1f} -> {stats["latent_after"]:.1f} '
      f'({stats["gap_closed"]:.1f}% of the gap closed)')
print(f'visible change: PSNR {stats["psnr"]:.2f} dB, max {stats["max_delta"]}/255 per channel')
print('wrote', OUT)

In [ ]:
# 5. Look at it, and magnify the difference 12x to convince yourself it is invisible
import numpy as np
from PIL import Image
a = np.asarray(img.resize(protected.size)).astype(np.int16)
b = np.asarray(protected).astype(np.int16)
diff = np.clip(128 + (b - a) * 12, 0, 255).astype(np.uint8)

w, h = protected.size
strip = Image.new('RGB', (w * 3 + 24, h), (16, 18, 24))
strip.paste(img.resize(protected.size), (0, 0))
strip.paste(protected, (w + 12, 0))
strip.paste(Image.fromarray(diff), (2 * w + 24, 0))
strip.save('comparison.png')
display(strip)   # left: original, middle: protected, right: difference x12

In [ ]:
# 6. Download the protected image. Keep the original too — it is your clean copy.
from google.colab import files
files.download(OUT)

## What to expect

Feed the protected file to an image-to-image or inpainting model and the edit should come out wrong: distorted, unrelated, or obviously synthetic. That is the attack working — the model's internal view of your photo has been flattened towards grey.

## Limits worth knowing

- **Re-compression erodes it.** Post the PNG where you can; if a site re-encodes heavily at low quality, part of the perturbation goes with it.
- **Cropping weakens it.** The perturbation is tied to the whole frame.
- **Newer models are more robust.** This was designed against Stable Diffusion-era LDMs; treat the protection as a speed bump that costs an attacker time and quality, not a lock.
- **Only protect your own photos**, or images you have the rights to protect.

The idea is from Salman, Khaddaj, Leclerc, Ilyas and Madry, arXiv:2302.06588. This notebook is an independent implementation — no code was copied from the reference repository.